# 1. Improt 

In [ ]:
import json
import time
import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import unquote
import os
from dotenv import load_dotenv
load_dotenv()

# 2. Ambil cookie

In [ ]:
# Konfigurasi & Inisialisasi
USERNAME = os.getenv("FASIH_USER")
PASSWORD = os.getenv("FASIH_PASS")
OAUTH_LOGIN_URL = "https://fasih-sm.bps.go.id/oauth_login.html"

def build_driver(headless=False):
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    if headless: options.add_argument("--headless=new")
    options.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_cdp_cmd("Network.enable", {})
    return driver

def get_authenticated_session(driver):
    session = requests.Session()
    for cookie in driver.get_cookies():
        session.cookies.set(cookie['name'], cookie['value'])
    bearer_token = driver.execute_script("return window.localStorage.getItem('token');")
    csrf_token = session.cookies.get('XSRF-TOKEN')
    auth_headers = {
        "User-Agent": driver.execute_script("return navigator.userAgent;"),
        "Accept": "application/json, text/plain, */*",
        "X-XSRF-TOKEN": unquote(csrf_token) if csrf_token else "",
        "Authorization": f"Bearer {bearer_token.replace('"', '')}" if bearer_token else ""
    }
    session.headers.update(auth_headers)
    return session

# Eksekusi Login
driver = build_driver(headless=False)
driver.get(OAUTH_LOGIN_URL)

# Alur Klik SSO & Input Login (Diringkas)
wait = WebDriverWait(driver, 30)
sso_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.login-button")))
driver.execute_script("arguments[0].click();", sso_btn)

wait.until(lambda d: "sso.bps.go.id" in d.current_url)
wait.until(EC.presence_of_element_located((By.ID, "username"))).send_keys(USERNAME)
pass_el = driver.find_element(By.ID, "password")
pass_el.send_keys(PASSWORD)
pass_el.send_keys(Keys.ENTER)

wait.until(lambda d: "fasih-sm.bps.go.id" in d.current_url)
print("Login Berhasil.")
time.sleep(5) # Tunggu sinkronisasi token ke localStorage

# Ambil Session untuk Scraping Cepat
api_session = get_authenticated_session(driver)

In [ ]:
driver.quit()

# 3. Print list of surveys

## def

In [ ]:
def fetch_list_surveys(session, page_size=100):
    # Sesuaikan URL dengan log (Pencacahan)
    url = "https://fasih-sm.bps.go.id/survey/api/v1/surveys/datatable?surveyType=Pencacahan"
    
    # Payload harus persis dengan log yang kamu kasih
    payload = {
        "pageNumber": 0,
        "pageSize": page_size,
        "sortBy": "CREATED_AT", # Harus Uppercase
        "sortDirection": "DESC",
        "keywordSearch": ""
    }
    
    response = session.post(url, json=payload)
    if response.status_code == 200:
        data = response.json()
        # Struktur Fasih: data -> content
        return data.get('data', {}).get('content', [])
    else:
        print(f"Gagal ambil list survei: {response.status_code}, Msg: {response.text}")
        return []

def process_surveys_to_df(rows):
    surveys_list = []
    for item in rows:
        surveys_list.append({
            "id": item.get("id"),
            "judul_survei": item.get("name"), # Sesuai key 'name' di response
            "unit": item.get("unit"),
            "dibuat_pada": item.get("createdAt")
        })
    return pd.DataFrame(surveys_list)

## implementasi

In [ ]:
# Jalankan setelah api_session di Bagian 2 siap
raw_surveys = fetch_list_surveys(api_session)

# Simpan ke variabel global surveys
surveys = raw_surveys 

# Tampilkan tabel
df_surveys = process_surveys_to_df(surveys)
print(f"Ditemukan {len(df_surveys)} survei.")

if not df_surveys.empty:
    # Menampilkan 20 data pertama
    display(df_surveys.head(20))
else:
    print("Cek kembali koneksi atau session.")

# 4. ambil data survey settings

## def

In [ ]:
def fetch_json(session, url):
    r = session.get(url, timeout=30)
    r.raise_for_status()
    return r.json().get("data", {})

def fetch_full_survey_settings_flat(session, survey_id):
    # 1. Tarik semua data mentah
    det = fetch_json(session, f"https://fasih-sm.bps.go.id/survey/api/v1/surveys/{survey_id}")
    per = fetch_json(session, f"https://fasih-sm.bps.go.id/survey/api/v1/survey-periods?surveyId={survey_id}")
    reg = fetch_json(session, f"https://fasih-sm.bps.go.id/region/api/v1/region-metadata?id={det.get('regionGroupId')}") if det.get('regionGroupId') else {}

    # 2. Ambil periode aktif
    act_per = next((p for p in per if p.get("isActive")), per[0] if per else {})

    # 3. Bangun Flat Dictionary (Metadata Inti)
    meta = {
        "judul": det.get("name"),
        "tipe": det.get("surveyType"),
        "mode": ", ".join([m.get("mode", "") for m in det.get("surveyModeList", [])]),
        "wilayah_ver": reg.get("groupName"),
        "level_wilayah": " > ".join([l.get("name", "") for l in reg.get("level", [])]),
        "jenis_panel": "Panel" if det.get("panelType") else "Non-Panel",
        "jenis_pencacah": "Banyak" if det.get("isMultiPencacah") else "Satu",
        "periode_aktif": act_per.get("name"),
        "tgl_mulai": act_per.get("startDate"),
        "tgl_selesai": act_per.get("endDate"),
        "id_periode": act_per.get("id")
    }
    
    # 4. Opsional: Tambahkan semua field asli jika ingin sangat lengkap
    # meta.update({f"raw_{k}": v for k, v in det.items() if not isinstance(v, (list, dict))})
    
    return [meta]

## implementasi


In [ ]:
# 1. Sinkronisasi Session (Jika belum)
api_session = get_authenticated_session(driver)

# 2. Ambil Metadata
survey_id_target = "8712a6fc-a996-4a8f-ad6f-56a278c19288"
settings = fetch_full_survey_settings_flat(api_session, survey_id_target)

# 3. Tampilkan (Key : Value)
item = settings[0]
for k, v in item.items():
    print(f"{k.ljust(20)} : {v}")

# 5. ambil data petugas

## def

In [ ]:
import requests
import pandas as pd
from urllib.parse import unquote

def fetch_api_with_selenium_auth(driver, api_url, payload):
    # 1. Ambil semua cookies
    all_cookies = driver.get_cookies()
    cookies_dict = {c['name']: c['value'] for c in all_cookies}
    
    # 2. Ambil CSRF Token dari Cookie (Kunci Utama)
    # Biasanya namanya 'XSRF-TOKEN'
    csrf_token = cookies_dict.get('XSRF-TOKEN')
    
    # 3. Ambil Bearer Token dari Local Storage
    bearer_token = driver.execute_script("return window.localStorage.getItem('token');")
    
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/plain, */*",
        "User-Agent": driver.execute_script("return navigator.userAgent;"),
        "Referer": "https://fasih-sm.bps.go.id/survey-collection/survey",
        "Origin": "https://fasih-sm.bps.go.id"
    }
    
    # Tambahkan CSRF ke Header jika ditemukan
    if csrf_token:
        # CSRF token di cookie seringkali ter-URL encode, kita decode dulu
        headers["X-XSRF-TOKEN"] = unquote(csrf_token)
    
    if bearer_token:
        headers["Authorization"] = f"Bearer {bearer_token.replace('"', '')}"

    # 4. Kirim request
    response = requests.post(api_url, json=payload, headers=headers, cookies=cookies_dict)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Gagal! Status: {response.status_code}, Response: {response.text}")
        return None

def process_to_dataframe(json_data):
    if not json_data or "data" not in json_data or "searchData" not in json_data["data"]:
        return pd.DataFrame()

    rows = []
    search_data = json_data["data"]["searchData"]
    
    for item in search_data:
        user = item.get("user", {})
        # Flattening data wilayah
        regions = [r.get("smallestRegionCode") for r in item.get("smallestRegionCodes", [])]
        
        rows.append({
            "Fullname": user.get("fullname"),
            "Email": user.get("email"),
            "Role": item.get("surveyRole", {}).get("description"),
            "Wilayah": ", ".join([str(r) for r in regions]) if regions else "-"
        })
    return pd.DataFrame(rows)

## implementasi

In [ ]:
# CHUNK 5 - DATA PETUGAS (Integrasi ke ID target)
# Pakai URL API Petugas yang mengandung survey_id_target
api_petugas = f"https://fasih-sm.bps.go.id/analytic/api/v2/survey-period-role-user/datatable?surveyPeriodId=39136966-8f3c-4a0c-915b-0f65eb223475&surveyRoleGroupId=b4f6b841-7a33-48a1-be2b-042d866e1938&surveyRoleId=6d392af0-fa8b-4d5e-8dff-cef279304e0a"

payload_petugas = {
    "pageNumber": 0, "pageSize": 100, "sortBy": "ID", "sortDirection": "ASC", "keywordSearch": ""
}

# Pakai session yang sudah dibuat di Chunk 4
resp_petugas = api_session.post(api_petugas, json=payload_petugas)

if resp_petugas.status_code == 200:
    data_petugas = resp_petugas.json()
    # Gunakan fungsi process_to_dataframe yang sudah kita buat tadi
    df_petugas = process_to_dataframe(data_petugas)
    print("\nData Petugas:")
    display(df_petugas)
else:
    print(f"Gagal ambil petugas: {resp_petugas.status_code}")

# 6. Ambil data survei

## def

In [ ]:
## Implementasi